# Assignment 3 
**Name:** Bhanavi
**Roll No:** 102313054

### Q1: K-Fold Cross Validation for Multiple Linear Regression (Least Square Error Fit)

#### Load the dataset and Implement 5- fold cross validation for multiple linear regression (using least square error fit).

##### a) Divide the dataset into input features (all columns except price) and output variable (price).

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

df = pd.read_csv("USA_Housing.csv")   

X = df.drop("Price", axis=1).values   
y = df["Price"].values             

##### b) Scale the values of input features.

In [2]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

##### c) Divide input and output features into five folds.

In [3]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

r2_scores = []
betas = []
preds_list = []

##### d) Run five iterations, in each iteration consider one-fold as test set and remaining four sets as training set. Find the beta (𝛽) matrix, predicted values, and R2_score or each iteration using least square error fit.

In [4]:
for train_idx, test_idx in kf.split(X_scaled):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    X_train_b = np.c_[np.ones(X_train.shape[0]), X_train]  
    X_test_b = np.c_[np.ones(X_test.shape[0]), X_test]

    beta = np.linalg.inv(X_train_b.T @ X_train_b) @ (X_train_b.T @ y_train)
    
    y_pred = X_test_b @ beta
    r2 = r2_score(y_test, y_pred)

    r2_scores.append(r2)
    betas.append(beta)
    preds_list.append(y_pred)

for i, r2 in enumerate(r2_scores):
    print(f"Fold {i+1} R2_score: {r2:.4f}")

best_idx = np.argmax(r2_scores)
print("\nBest Fold:", best_idx+1)
print("Best Beta Matrix:\n", betas[best_idx])

Fold 1 R2_score: 0.9180
Fold 2 R2_score: 0.9146
Fold 3 R2_score: 0.9116
Fold 4 R2_score: 0.9193
Fold 5 R2_score: 0.9244

Best Fold: 5
Best Beta Matrix:
 [1.23161736e+06 2.30225051e+05 1.63956839e+05 1.21115120e+05
 7.83467170e+02 1.50662447e+05]


##### e) Use the best value of (𝛽) matrix (for which R2_score is maximum), to train the regressor for 70% of data and test the performance for remaining 30% data.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42)

X_train_b = np.c_[np.ones(X_train.shape[0]), X_train]
X_test_b = np.c_[np.ones(X_test.shape[0]), X_test]

best_beta = betas[best_idx]
y_pred_final = X_test_b @ best_beta
print("\nFinal R2 Score (70/30 split):", r2_score(y_test, y_pred_final))


Final R2 Score (70/30 split): 0.9147458156636434


### Q2: Concept of Validation set for Multiple Linear Regression (Gradient Descent Optimization)

##### Consider the same dataset of Q1, rather than dividing the dataset into five folds, divide the dataset into training set (56%), validation set (14%), and test set (30%). Consider four different values of learning rate i.e. {0.001,0.01,0.1,1}. Compute the values of regression coefficients for each value of learning rate after 1000 iterations. For each set of regression coefficients, compute R2_score for validation and test set and find the best value of regression coefficients. 

In [6]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.2, random_state=42)  # 0.2 of 70% ≈ 14%

X_train_b = np.c_[np.ones(X_train.shape[0]), X_train]
X_val_b = np.c_[np.ones(X_val.shape[0]), X_val]
X_test_b = np.c_[np.ones(X_test.shape[0]), X_test]

def gradient_descent(X, y, lr, iterations=1000):
    m, n = X.shape
    beta = np.zeros(n)
    
    for _ in range(iterations):
        y_pred = X @ beta
        error = y_pred - y
        gradient = (1/m) * (X.T @ error)
        beta -= lr * gradient
    
    return beta

learning_rates = [0.001, 0.01, 0.1, 1]
results = {}

for lr in learning_rates:
    beta = gradient_descent(X_train_b, y_train, lr, 1000)
    
    y_val_pred = X_val_b @ beta
    y_test_pred = X_test_b @ beta
    
    r2_val = r2_score(y_val, y_val_pred)
    r2_test = r2_score(y_test, y_test_pred)
    
    results[lr] = (beta, r2_val, r2_test)

for lr, (beta, r2_val, r2_test) in results.items():
    print(f"LR={lr} -> R2 Validation: {r2_val:.4f}, R2 Test: {r2_test:.4f}")

best_lr = max(results, key=lambda lr: results[lr][1])  # max validation score
print("\nBest Learning Rate:", best_lr)
print("Best Beta Matrix:\n", results[best_lr][0])

LR=0.001 -> R2 Validation: -0.8125, R2 Test: -0.9914
LR=0.01 -> R2 Validation: 0.9098, R2 Test: 0.9147
LR=0.1 -> R2 Validation: 0.9098, R2 Test: 0.9148
LR=1 -> R2 Validation: 0.9098, R2 Test: 0.9148

Best Learning Rate: 0.01
Best Beta Matrix:
 [1232562.51254919  230048.76664688  163686.93503606  121406.94107918
    3117.47363933  150655.97459714]


### Q3: Pre-processing and Multiple Linear Regression

##### 1. Load the dataset with following column names ["symboling", "normalized_losses","make", "fuel_type", "aspiration","num_doors", "body_style", "drive_wheels", "engine_location", "wheel_base", "length", "width", "height", "curb_weight", "engine_type", "num_cylinders", "engine_size", "fuel_system", "bore", "stroke", "compression_ratio", "horsepower", "peak_rpm", "city_mpg", "highway_mpg", "price"] and replace all ? values with NaN.

In [9]:
import pandas as pd
import numpy as np

columns = ["symboling", "normalized_losses", "make", "fuel_type", "aspiration",
           "num_doors", "body_style", "drive_wheels", "engine_location", "wheel_base",
           "length", "width", "height", "curb_weight", "engine_type", "num_cylinders",
           "engine_size", "fuel_system", "bore", "stroke", "compression_ratio",
           "horsepower", "peak_rpm", "city_mpg", "highway_mpg", "price"]

url = r"D:\Documents\BE_3\5_UML501 Machine Learning (4)\UML501 Repository\UML501\imports-85.data.txt"
df = pd.read_csv(url, names=columns)

df.replace("?", np.nan, inplace=True)

print(df.head())
print(df.info())


   symboling normalized_losses         make fuel_type aspiration num_doors  \
0          3               NaN  alfa-romero       gas        std       two   
1          3               NaN  alfa-romero       gas        std       two   
2          1               NaN  alfa-romero       gas        std       two   
3          2               164         audi       gas        std      four   
4          2               164         audi       gas        std      four   

    body_style drive_wheels engine_location  wheel_base  ...  engine_size  \
0  convertible          rwd           front        88.6  ...          130   
1  convertible          rwd           front        88.6  ...          130   
2    hatchback          rwd           front        94.5  ...          152   
3        sedan          fwd           front        99.8  ...          109   
4        sedan          4wd           front        99.4  ...          136   

   fuel_system  bore  stroke compression_ratio horsepower  peak_rpm 

##### 2. Replace all NaN values with central tendency imputation. Drop the rows with NaN values in price column.

In [10]:
numeric_cols = ["normalized_losses", "wheel_base", "length", "width", "height", 
                "curb_weight", "engine_size", "bore", "stroke", "compression_ratio", 
                "horsepower", "peak_rpm", "city_mpg", "highway_mpg", "price"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df.dropna(subset=["price"], inplace=True)

for col in numeric_cols:
    df[col].fillna(df[col].mean(), inplace=True)

categorical_cols = ["make", "fuel_type", "aspiration", "num_doors", "body_style",
                    "drive_wheels", "engine_location", "engine_type", "num_cylinders", "fuel_system"]

for col in categorical_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

C:\Users\bhana\AppData\Local\Temp\ipykernel_8964\2896174061.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mean(), inplace=True)
C:\Users\bhana\AppData\Local\Temp\ipykernel_8964\2896174061.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example

##### 3. There are 10 columns in the dataset with non-numeric values. Convert these values to numeric values using following scheme:
###### (i) For “num_doors” and “num_cylinders”: convert words (number names) to figures for e.g., two to 2
###### (ii) For "body_style", "drive_wheels": use dummy encoding scheme
###### (iii) For “make”, “aspiration”, “engine_location”,fuel_type: use label encoding scheme
###### (iv) For fuel_system: replace values containing string pfi to 1 else all values to 0.
###### (v) For engine_type: replace values containing string ohc to 1 else all values to 0.


In [ ]:
from sklearn.preprocessing import LabelEncoder

word_to_num = {"two":2, "three":3, "four":4, "five":5, "six":6, "eight":8, "twelve":12}
df["num_doors"].replace({"two":2, "four":4}, inplace=True)
df["num_cylinders"].replace(word_to_num, inplace=True)

df = pd.get_dummies(df, columns=["body_style", "drive_wheels"], drop_first=True)

label_cols = ["make", "aspiration", "engine_location", "fuel_type"]
le = LabelEncoder()
for col in label_cols:
    df[col] = le.fit_transform(df[col])

df["fuel_system"] = df["fuel_system"].apply(lambda x: 1 if "pfi" in x else 0)

df["engine_type"] = df["engine_type"].apply(lambda x: 1 if "ohc" in x else 0)

C:\Users\bhana\AppData\Local\Temp\ipykernel_8964\4221289627.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["num_doors"].replace({"two":2, "four":4}, inplace=True)
C:\Users\bhana\AppData\Local\Temp\ipykernel_8964\4221289627.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["num_cylinders

##### 4. Divide the dataset into input features (all columns except price) and output variable (price). Scale all input features.

In [12]:
from sklearn.preprocessing import StandardScaler

X = df.drop("price", axis=1).values
y = df["price"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

##### 5. Train a linear regressor on 70% of data (using inbuilt linear regression function ofPython) and test its performance on remaining 30% of data.

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("R2 score (original features):", r2_score(y_test, y_pred))

R2 score (original features): 0.87327756820863


##### 6. Reduce the dimensionality of the feature set using inbuilt PCA decomposition and then again train a linear regressor on 70% of reduced data (using inbuilt linear regression function of Python). Does it lead to any performance improvement on test set? 

In [14]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled)

Xp_train, Xp_test, yp_train, yp_test = train_test_split(X_pca, y, test_size=0.3, random_state=42)

model_pca = LinearRegression()
model_pca.fit(Xp_train, yp_train)
yp_pred = model_pca.predict(Xp_test)

print("R2 score (PCA features):", r2_score(yp_test, yp_pred))

R2 score (PCA features): 0.8611839960452383
